In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

def count_kmer_with_revcomp(dna_seq, k):
    kmer_count = defaultdict(int)
    seq_len = len(dna_seq)
    
    seq = Seq(dna_seq)
    rev_comp_seq = str(seq.reverse_complement())
    
    for i in range(seq_len - k + 1):
        kmer = dna_seq[i:i+k]
        kmer_count[kmer] += 1
    
    for i in range(seq_len - k + 1):
        kmer = rev_comp_seq[i:i+k]
        kmer_count[kmer] += 1
        
    return dict(kmer_count)

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import random
from tqdm import tqdm
from Bio import SeqIO
from Bio.Seq import Seq
from collections import defaultdict

random_seed = 42

def generate_random_region(target_size, contig_size, random_seed=random_seed):
    random.seed(random_seed)
    max_start = max(contig_size - target_size, 0)
    start = random.randint(0, max_start)
    end = min(start + target_size, contig_size)
    return int(start), int(end)
    
for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    replicon_data = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    sub_replicon = replicon_data[replicon_data['category-pident_90'] != 'typical chromosome']
    typical_chr = replicon_data[replicon_data['category-pident_90'] == 'typical chromosome']
    sub_replicon_sample = sub_replicon.sample(n=1000, random_state=random_seed, replace=True).reset_index(drop=True)
    typical_chr_sample = typical_chr.sample(n=len(sub_replicon_sample), random_state=random_seed, replace=True).reset_index(drop=True)

    kmer_dir = f'{folder.replace('statistics_records','kmer_chr_frag_samples')}/fragment_records'
    os.makedirs(kmer_dir, exist_ok=True)
    with tqdm(total = len(typical_chr_sample), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for i in range(len(typical_chr_sample)):
            target_size = sub_replicon_sample.loc[i, 'size']
            acc_n, contig = typical_chr_sample.loc[i, 'accession'].split('-')
            contig_size = typical_chr_sample.loc[i, 'size']
            
            start, end = generate_random_region(target_size, contig_size, random_seed=i)
            handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
            seq_record = SeqIO.parse(handle, 'genbank')
            for record in seq_record:
                if record.id != contig:
                    continue
                dna_sequence = str(record.seq)[start:end]
                kmer_info = {}
                for k in range(5):
                    res = count_kmer_with_revcomp(dna_sequence, k+1)
                    kmer_info[f'{k+1}-mer'] = res
                file = open(f'{kmer_dir}/{acc_n}-{record.id}-{start}-{end}.txt', 'w+')
                file.write(str(kmer_info))
                file.close()
            pbar.update(1)

Escherichia: 100%|██████████████████████████████████████████████| 1.00k/1.00k [10:13<00:00, 1.63B/s]
Klebsiella: 100%|███████████████████████████████████████████████| 1.00k/1.00k [11:31<00:00, 1.45B/s]
Staphylococcus: 100%|███████████████████████████████████████████| 1.00k/1.00k [04:40<00:00, 3.56B/s]
Pseudomonas: 100%|██████████████████████████████████████████████| 1.00k/1.00k [14:06<00:00, 1.18B/s]
Bacillus: 100%|█████████████████████████████████████████████████| 1.00k/1.00k [10:27<00:00, 1.59B/s]
Salmonella: 100%|███████████████████████████████████████████████| 1.00k/1.00k [10:03<00:00, 1.66B/s]
Streptococcus: 100%|████████████████████████████████████████████| 1.00k/1.00k [03:32<00:00, 4.72B/s]
Streptomyces: 100%|█████████████████████████████████████████████| 1.00k/1.00k [18:31<00:00, 1.11s/B]
Acinetobacter: 100%|████████████████████████████████████████████| 1.00k/1.00k [07:07<00:00, 2.34B/s]
Enterococcus: 100%|█████████████████████████████████████████████| 1.00k/1.00k [06:17<00:00,